In [50]:
import pandas as pd
import numpy as np
from calendar import monthrange
from sqlalchemy import create_engine, text
from pandas.tseries.offsets import MonthEnd

def create_date_column(df):
    # 각 월의 마지막 날짜 계산 함수
    def get_last_day_of_month(year, month):
        return monthrange(int(year), int(month))[1]

    # 날짜 열 생성
    df['Date'] = df.apply(
        lambda row: pd.to_datetime(
            f"{int(row['회계연도'])}-{int(row['결산월'])}-{get_last_day_of_month(row['회계연도'], row['결산월'])}"
        ),
        axis=1
    )

    # 주기가 분기형이면, 날짜를 해당 분기 말일로 직접 지정
    if '주기' in df.columns:
        quarter_map = {
            '1Q': '-03-31',
            '2Q': '-06-30',
            '3Q': '-09-30',
            '4Q': '-12-31'
        }

        def override_to_quarter_end(row):
            q = str(row['주기']).strip()
            y = int(row['회계연도'])
            return pd.to_datetime(f"{y}{quarter_map[q]}") if q in quarter_map else row['Date']

        df['Date'] = df.apply(override_to_quarter_end, axis=1)

    return df


# ======================
# 1. 데이터 로드 및 전처리
# ======================
#  C:\Users\82108\OneDrive\바탕 화면\investment\data\raw_data : 여기로 경로 이동 함
path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\raw_data\is_bs_cf_Dataguide_2025_3Q.xlsx"
sheet_name = "KOSDAQ"

raw_df = pd.read_excel(path, sheet_name=sheet_name)

# 기준 행
row_header_1 = 8  # 항목명
row_header_2 = 9  # 단위, 코드, 분류 등

# 복합 컬럼명 생성
def combine_headers(col):
    item = str(raw_df.loc[row_header_1, col]) if pd.notna(raw_df.loc[row_header_1, col]) else ""
    unit = str(raw_df.loc[row_header_2, col]) if pd.notna(raw_df.loc[row_header_2, col]) else ""
    combined = f"{unit.strip()}: {item.strip()}" if unit and item else item or unit
    return combined if combined else col

# 새로운 컬럼 리스트 생성
new_columns = [combine_headers(col) for col in raw_df.columns]

# 컬럼명 적용
df_cleaned = raw_df.copy()
df_cleaned.columns = new_columns
df_cleaned = df_cleaned.iloc[10:].reset_index(drop=True)

# "Name"을 "company_name"으로 변경
if "Name" in df_cleaned.columns:
    df_cleaned = df_cleaned.rename(columns={"Name": "company_name"})

print("=== 전처리 후 컬럼 목록 (처음 30개) ===")
for i, col in enumerate(df_cleaned.columns[:30], 1):
    print(f"{i}. {col}")

# ======================
# 2. 실제 컬럼명 패턴 분석
# ======================
print("\n=== Symbol/Code 관련 컬럼 검색 ===")
for col in df_cleaned.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['symbol', 'code', '코드', '종목']):
        print(f"  - {col}")

print("\n=== 년도 관련 컬럼 검색 ===")
for col in df_cleaned.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['year', '년', '연도', '회계']):
        print(f"  - {col}")

print("\n=== 분기/주기 관련 컬럼 검색 ===")
for col in df_cleaned.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['quarter', 'q', '분기', '주기', 'period']):
        print(f"  - {col}")

# ======================
# 3. 수동으로 컬럼 지정 (위 출력 결과를 보고 수정)
# ======================
# 실제 컬럼명을 확인한 후 직접 지정
col_symbol = 'Symbol'  # 실제 컬럼명으로 수정 필요
col_year = None  # 위 출력 결과를 보고 수정
col_period = None  # 위 출력 결과를 보고 수정

# 아래는 자동 검색 시도
for col in df_cleaned.columns:
    # Symbol 찾기
    if col_symbol == 'Symbol' and col in df_cleaned.columns:
        col_symbol = col
        break
    elif 'Symbol' in str(col) or 'CODE' in str(col).upper():
        col_symbol = col
        break

# 회계년 찾기 (더 관대한 조건)
for col in df_cleaned.columns:
    col_str = str(col)
    if '년' in col_str and '천원' not in col_str:
        col_year = col
        break

# 주기 찾기 (더 관대한 조건)
for col in df_cleaned.columns:
    col_str = str(col)
    if ('분기' in col_str or '주기' in col_str or 'Q' in col_str) and '천원' not in col_str:
        col_period = col
        break

# 재무 지표 컬럼 찾기
def find_column(df, keyword):
    for col in df.columns:
        if keyword in str(col) and '(천원)' in str(col):
            return col
    return None

col_sales = find_column(df_cleaned, '매출액')
col_gross = find_column(df_cleaned, '매출총이익')
col_operating = find_column(df_cleaned, '영업이익')
col_continuing = find_column(df_cleaned, '계속사업이익')
col_net_income = find_column(df_cleaned, '당기순이익')
col_noncurrent_liab = find_column(df_cleaned, '비유동부채')

# 자본 컬럼 찾기
col_equity = None
for col in df_cleaned.columns:
    if "자본총계" in str(col) and "지배" in str(col) and "(천원)" in str(col):
        col_equity = col
        break

# ======================
# 4. 결과 확인
# ======================
print("\n=== 컬럼 매칭 결과 ===")
col_mapping = {
    '종목코드': col_symbol,
    '회계년': col_year,
    '주기': col_period,
    '매출액': col_sales,
    '매출총이익': col_gross,
    '영업이익': col_operating,
    '계속사업이익': col_continuing,
    '당기순이익': col_net_income,
    '비유동부채': col_noncurrent_liab,
    '자본': col_equity
}

for name, col in col_mapping.items():
    status = "✓" if col is not None else "✗"
    print(f"{status} {name}: {col}")

# Missing 체크
missing_cols = [name for name, col in col_mapping.items() if col is None]

if missing_cols:
    print(f"\n⚠️ 다음 컬럼을 찾지 못했습니다: {', '.join(missing_cols)}")
    print("\n수동으로 컬럼명을 확인하고 코드에서 직접 지정해주세요.")
    print("예: col_year = '실제컬럼명'")

    # 일단 계속 진행 (선택적)
    input("\n계속하려면 Enter를 누르세요 (에러 발생 가능)...")

# ======================
# 5. 숫자형으로 변환
# ======================
numeric_cols = [col_sales, col_gross, col_operating, col_continuing,
                col_net_income, col_noncurrent_liab, col_equity]
numeric_cols = [col for col in numeric_cols if col is not None]

for col in numeric_cols:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# ======================
# 6. 데이터 정렬 및 YoY 계산
# ======================
if col_symbol and col_year and col_period:
    df_cleaned = df_cleaned.sort_values(by=[col_symbol, col_year, col_period])

    # YoY 증가율 계산
    if col_sales:
        df_cleaned["YoY_매출액"] = df_cleaned.groupby(col_symbol)[col_sales].pct_change(periods=4, fill_method=None)
    if col_gross:
        df_cleaned["YoY_매출총이익"] = df_cleaned.groupby(col_symbol)[col_gross].pct_change(periods=4, fill_method=None)
    if col_operating:
        df_cleaned["YoY_영업이익"] = df_cleaned.groupby(col_symbol)[col_operating].pct_change(periods=4, fill_method=None)
    if col_continuing:
        df_cleaned["YoY_계속사업이익"] = df_cleaned.groupby(col_symbol)[col_continuing].pct_change(periods=4, fill_method=None)
    if col_net_income:
        df_cleaned["YoY_당기순이익"] = df_cleaned.groupby(col_symbol)[col_net_income].pct_change(periods=4, fill_method=None)
else:
    print("\n⚠️ 정렬 및 YoY 계산을 건너뜁니다 (필수 컬럼 부족)")

# ======================
# 7. 수익성 비율 계산
# ======================
if col_gross and col_sales:
    df_cleaned["매출총이익률"] = df_cleaned[col_gross] / df_cleaned[col_sales]
if col_operating and col_sales:
    df_cleaned["영업이익률"] = df_cleaned[col_operating] / df_cleaned[col_sales]
if col_net_income and col_sales:
    df_cleaned["순이익률"] = df_cleaned[col_net_income] / df_cleaned[col_sales]
if col_noncurrent_liab and col_equity:
    df_cleaned["비유동부채비율"] = df_cleaned[col_noncurrent_liab] / df_cleaned[col_equity]

# ======================
# 8. 결과 미리보기
# ======================
display_cols = []
if col_symbol:
    display_cols.append(col_symbol)
if col_year:
    display_cols.append(col_year)
if col_period:
    display_cols.append(col_period)
if col_sales:
    display_cols.append(col_sales)

display_cols += [col for col in ["YoY_매출액", "YoY_영업이익", "영업이익률", "순이익률", "비유동부채비율"]
                 if col in df_cleaned.columns]

if display_cols:
    print("\n=== 결과 미리보기 ===")
    print(df_cleaned[display_cols].head(20))

    print("\n=== 통계 정보 ===")
    print(f"총 행 수: {len(df_cleaned)}")
    if col_symbol:
        print(f"고유 종목 수: {df_cleaned[col_symbol].nunique()}")

=== 전처리 후 컬럼 목록 (처음 30개) ===
1. 코드
2. 코드명
3. 결산월
4. 회계연도
5. 분기
6. 매출액(천원)
7. 매출총이익(천원)
8. 영업이익(천원)
9. 계속사업이익(천원)
10. 당기순이익(천원)
11. 총포괄이익(지배)(천원)
12. 유형자산감가상각비(천원)
13. 연구개발비(천원)
14. 자산총계(천원)
15. 유동자산(천원)
16. 당좌자산(천원)
17. 매출채권및기타채권(천원)
18. 재고자산(천원)
19. 투자부동산(천원)
20. 비유동부채(천원)
21. 자본총계(천원)
22. 자본총계(지배)(천원)
23. 영업활동으로인한현금흐름(천원)
24. 영업활동으로인한현금흐름(TTM)(천원)
25. 영업활동으로인한현금흐름(평균)(천원)
26. 배당금지급(영업,투자,재무)(천원)

=== Symbol/Code 관련 컬럼 검색 ===
  - 코드
  - 코드명

=== 년도 관련 컬럼 검색 ===
  - 회계연도

=== 분기/주기 관련 컬럼 검색 ===
  - 분기

=== 컬럼 매칭 결과 ===
✓ 종목코드: 코드
✗ 회계년: None
✓ 주기: 분기
✓ 매출액: 매출액(천원)
✓ 매출총이익: 매출총이익(천원)
✓ 영업이익: 영업이익(천원)
✓ 계속사업이익: 계속사업이익(천원)
✓ 당기순이익: 당기순이익(천원)
✓ 비유동부채: 비유동부채(천원)
✓ 자본: 자본총계(지배)(천원)

⚠️ 다음 컬럼을 찾지 못했습니다: 회계년

수동으로 컬럼명을 확인하고 코드에서 직접 지정해주세요.
예: col_year = '실제컬럼명'

⚠️ 정렬 및 YoY 계산을 건너뜁니다 (필수 컬럼 부족)

=== 결과 미리보기 ===
         코드   분기  매출액(천원)  영업이익률  순이익률  비유동부채비율
0   A196170  1QR      NaN    NaN   NaN      NaN
1   A196170  2QR      NaN    NaN   NaN      NaN
2   A196170  3QR      NaN    NaN   NaN  

In [51]:
df_cleaned = df_cleaned.dropna(subset=['회계연도', '결산월'])

In [52]:
df_cleaned = create_date_column(df_cleaned)

In [54]:
def convert_to_long_format_v2(df_cleaned):
    """
    컬럼 인덱스로 접근하여 변환 + 안전한 정수형 데이터 처리
    """
    import numpy as np

    def create_date(year, quarter):
        quarter_end_months = {
            '1QR': '03-31', '2QR': '06-30',
            '3QR': '09-30', '4QR': '12-31'
        }
        return f"{year}-{quarter_end_months[quarter]}"

    # 날짜 생성
    if 'date' not in df_cleaned.columns:
        df_cleaned['date'] = df_cleaned.apply(
            lambda row: create_date(row['회계연도'], row['분기']),
            axis=1
        )

    # 작업용 데이터프레임 생성
    df_temp = df_cleaned.copy()

    # symbol과 company_name 컬럼 추가
    df_temp['symbol'] = df_temp.iloc[:, 0]
    df_temp['company_name'] = df_temp.iloc[:, 1]

    # 기본 컬럼
    base_cols = ['symbol', 'company_name', 'date']

    # 매출액 이후, Date/date 제외한 지표 컬럼들
    indicator_start_idx = df_cleaned.columns.get_loc('매출액(천원)')
    indicator_cols = [
        col for col in df_cleaned.columns[indicator_start_idx:]
        if col not in ['Date', 'date']
    ]

    print(f"변환할 지표 수: {len(indicator_cols)}")

    # Long format 변환
    df_long = pd.melt(
        df_temp[base_cols + indicator_cols],
        id_vars=base_cols,
        value_vars=indicator_cols,
        var_name='indicator',
        value_name='value'
    )

    # 안전한 정수 변환 함수
    def safe_int_convert(val):
        if pd.isna(val):
            return val
        try:
            if np.isfinite(val) and val == int(val):
                return int(val)
            return val
        except (ValueError, TypeError, OverflowError):
            return val

    # value 컬럼에 적용
    df_long['value'] = df_long['value'].apply(safe_int_convert)

    # 정렬
    df_long = df_long.sort_values(['symbol', 'date', 'indicator']).reset_index(drop=True)

    print(f"\n변환 완료: {len(df_long):,} rows")
    print(f"고유 종목: {df_long['symbol'].nunique()}")
    print(f"고유 날짜: {df_long['date'].nunique()}")
    print(f"고유 지표: {df_long['indicator'].nunique()}")

    return df_long


# 사용
df_long = convert_to_long_format_v2(df_cleaned)
print(df_long.head(20))


변환할 지표 수: 25

변환 완료: 3,775,200 rows
고유 종목: 1716
고유 날짜: 88
고유 지표: 25
     symbol company_name        date              indicator         value
0   A000250        삼천당제약  2004-03-31             계속사업이익(천원)  2.352535e+06
1   A000250        삼천당제약  2004-03-31              당기순이익(천원)  2.352535e+06
2   A000250        삼천당제약  2004-03-31               당좌자산(천원)  5.460507e+07
3   A000250        삼천당제약  2004-03-31                매출액(천원)  1.261910e+07
4   A000250        삼천당제약  2004-03-31          매출채권및기타채권(천원)  3.406700e+07
5   A000250        삼천당제약  2004-03-31              매출총이익(천원)  7.951428e+06
6   A000250        삼천당제약  2004-03-31                 매출총이익률  6.301105e-01
7   A000250        삼천당제약  2004-03-31    배당금지급(영업,투자,재무)(천원)           NaN
8   A000250        삼천당제약  2004-03-31              비유동부채(천원)  5.124673e+06
9   A000250        삼천당제약  2004-03-31                비유동부채비율  1.022529e-01
10  A000250        삼천당제약  2004-03-31                   순이익률  1.864265e-01
11  A000250        삼천당제약  2004-03-31        

In [55]:
# 사용
df_long = convert_to_long_format_v2(df_cleaned)

# inf → NaN, 그 뒤 객체형 변환 보정
df_long = df_long.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

# 또는 특정 열만 안전하게 처리
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long = df_long.dropna(subset=["value"])

new_col = ['symbol', 'company_name', 'date', 'indicator', 'value']
df_long.columns = new_col

fs_value_df = df_long

변환할 지표 수: 25

변환 완료: 3,775,200 rows
고유 종목: 1716
고유 날짜: 88
고유 지표: 25


In [56]:

def upload_fs_data_to_db(df_long, db_info, table_name="korea_fs_data", chunk_size=1000):
    """
    DB에 long format 재무 데이터를 테이블 생성 후 업로드함.
    기존 테이블은 없다고 가정하고 새로 생성함.
    """
    # ✅ 날짜 및 결측치 처리
    df_long['date'] = pd.to_datetime(df_long['date'])
    df_long = df_long.replace([np.inf, -np.inf], np.nan)
    df_long = df_long.where(pd.notnull(df_long), None)

    # ✅ DB 연결
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    conn = engine.raw_connection()
    cursor = conn.cursor()

    # ✅ 테이블 생성 쿼리
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
        `symbol` VARCHAR(20),
        `company_name` VARCHAR(50),
        `date` DATE,
        `indicator` VARCHAR(100),
        `value` DOUBLE,
        PRIMARY KEY (`symbol`, `date`, `indicator`)
    );
    """
    cursor.execute(create_table_sql)
    conn.commit()

    # ✅ 데이터 INSERT 쿼리
    insert_sql = f"""
    INSERT INTO `{table_name}` (`symbol`, `company_name`, `date`, `indicator`, `value`)
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        `company_name` = VALUES(`company_name`),
        `value` = VALUES(`value`);
    """

    # ✅ 튜플 리스트로 변환
    rows = df_long[["symbol", "company_name", "date", "indicator", "value"]].values.tolist()

    # ✅ Chunk 단위로 업로드
    for i in range(0, len(rows), chunk_size):
        chunk = rows[i:i+chunk_size]
        cursor.executemany(insert_sql, chunk)
        conn.commit()

    cursor.close()
    conn.close()
    print(f"✅ 총 {len(df_long)}개 row 업로드 완료 (중복은 자동 업데이트됨)")


    def convert_to_long_format_v2(df):
        # 제거할 열
        drop_cols = ["결산월", "회계년", "주기"]

        # ID 변수 (고정값 유지할 열들)
        id_vars = ["Symbol", "company_name", "Date"]

        # 나머지는 전부 indicator 대상 열로 melt 처리
        value_vars = [col for col in df.columns if col not in id_vars + drop_cols]

        # melt 실행
        df_long = pd.melt(df,
                          id_vars=id_vars,
                          value_vars=value_vars,
                          var_name="indicator",
                          value_name="value")

        return df_long

In [57]:
db_info = {
    "user": 'stox7412',
    "password": 'Apt106503!~',
    "host": '192.168.0.230',
    # 'host': 'hystox74.synology.me',
    "port": 3307,
    "database": "investar"
}

upload_fs_data_to_db(fs_value_df, db_info)

✅ 총 1971380개 row 업로드 완료 (중복은 자동 업데이트됨)
